# Figures — from sequenced library to final assembly

Two figures, built from the stage tables so they cannot disagree with them:

| file | what it answers |
|---|---|
| `flow_attrition.svg` | why 316 sequenced libraries became 294 genomes |
| `route_provenance.svg` | how each of those 294 was actually produced |

They are deliberately separate. "Why 294" cuts the cohort by fate; "how were they
made" cuts the same 294 by production route. Crossing the two would mean cutting
the surviving channel on two incompatible axes at once.

Every count is derived here — nothing is typed in. The notebook asserts mass
conservation at every node before drawing, and measures each finished figure for
label collisions before writing it.

In [1]:
import html
from pathlib import Path

import pandas as pd

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships
HERE = Path(".")                      # the two SVGs are written beside this notebook

internal = pd.read_csv(TABLES / "internal_qc_summary.tsv", sep="\t")
external = pd.read_csv(TABLES / "external_qc_summary.tsv", sep="\t")
final    = pd.read_csv(TABLES / "final_qc_summary.tsv", sep="\t")
roster   = pd.read_csv(TABLES / "KASPAH_QC_outcome_v2.tsv", sep="\t")

print(f"{len(roster)} libraries \u00b7 {len(external)} into external QC \u00b7 {len(final)} final genomes")


316 libraries · 303 into external QC · 294 final genomes


## The ledger

Each of the 316 libraries gets a **retrospective fate band** — the class it ends up
in — assigned at entry. Every link in the Sankey is decomposed by these classes,
which is what keeps the surviving channel legible instead of one flat slab.

| band | | |
|---|---|---|
| `S1` | straight through | kept, no intervention, reproduces the reference |
| `S2` | sound, reference disagrees | kept and flagged — the metadata is the weaker evidence |
| `S3` | needed intervention | kept, but took the decontamination or the repolish detour |
| `D1` | duplicate morphology | dropped, **no isolate lost** — `.1`/`.2` are two colony picks of one isolate |
| `D2` | isolate lost | dropped, and nothing else in the cohort represents that isolate |

`D1` and `D2` are kept apart because they are not the same event. Dropping one of a
pair of colony picks is the design working as intended; only the four in `D2` cost
the collection an isolate.

In [2]:
BANDS = ["S1", "S2", "S3", "D1", "D2"]
LABEL = {"S1": "straight through", "S2": "sound, reference disagrees",
         "S3": "needed intervention", "D1": "duplicate morphology",
         "D2": "isolate lost"}
STAGE_OF = {"internal_qc": "05", "asm_fixing": "06", "external_qc": "07"}

def fate_table():
    f = final.set_index("library")
    intervened = set(f.index[(f.read_prep != "standard")
                             | f.assembly_route.str.contains("repolish")])
    flagged = set(f.index[~f.matches_reference])
    kept = set(f.index)
    rows = []
    for s, iso, dec, stage in roster[["sample", "isolate", "qc_decision", "qc_stage"]].itertuples(index=False):
        if s in kept:
            band = "S3" if s in intervened else ("S2" if s in flagged else "S1")
            rows.append((s, iso, band, None))
        else:
            band = "D1" if dec in ("FAIL_VARIANT", "DROP_VARIANT") else "D2"
            rows.append((s, iso, band, STAGE_OF[stage]))
    return pd.DataFrame(rows, columns=["sample", "isolate", "band", "exit"])

fate = fate_table()
n = fate.band.value_counts()
fate.groupby(["band", "exit"], dropna=False).size().to_frame("n")

n
band exit     
D1   05     10
     07      8
D2   06      3
     07      1
S1   NaN   268
S2   NaN    15
S3   NaN    11

In [3]:
# ── mass conservation, node by node ───────────────────────────────────────────
kept = [b for b in BANDS if b.startswith("S")]
drop = [b for b in BANDS if b.startswith("D")]

assert len(fate) == 316, len(fate)
assert n.sum() == 316
assert sum(n[b] for b in kept) == len(final) == 294
assert sum(n[b] for b in drop) == 22
assert (n["S1"], n["S2"], n["S3"], n["D1"], n["D2"]) == (268, 15, 11, 18, 4)

# the three survivor classes partition the 294 with no overlap
f = final.set_index("library")
interv = (f.read_prep != "standard") | f.assembly_route.str.contains("repolish")
assert not (interv & ~f.matches_reference).any(), "a genome is both flagged and intervened"

# stage-by-stage
exits = fate.groupby(["exit", "band"]).size()
assert exits[("05", "D1")] == 10 and exits[("07", "D1")] == 8
assert exits[("06", "D2")] == 3 and exits[("07", "D2")] == 1
assert set(exits.index.get_level_values(0)) == {"05", "06", "07"}

into_external = 316 - 10                       # 05 removes 10, and 06's 3 leave later
assert into_external - 3 == len(external) == 303
assert internal.internal_qc_decision.str.startswith("REDO").sum() == 9
assert internal.internal_qc_decision.str.startswith("PASS").sum() == 297
assert (internal.internal_qc_decision == "FAIL_VARIANT").sum() == 10

# isolates are NOT the flowing unit: 18 of the 22 discards cost no isolate
iso_entry, iso_final = roster.isolate.nunique(), final.isolate.nunique()
iso_external = external.isolate.nunique()
assert (iso_entry, iso_external, iso_final) == (298, 295, 294)
assert iso_final == len(final), "one library per isolate at the end"

# the intervened band threads both detours; INF205 is the only library in both
d06 = set(f.index[f.read_prep != "standard"]) | set(f.index[f.assembly_route.str.startswith("autocycler")])
d08 = set(f.index[f.assembly_route.str.contains("repolish")])
assert len(d06) == 5 and len(d08) == 7 and len(d06 | d08) == n["S3"] == 11
assert d06 & d08 == {"INF205"}

print(f"libraries  316 = {' + '.join(f'{n[b]} {b}' for b in BANDS)}")
print(f"isolates   {iso_entry} entry -> {iso_external} into external QC -> {iso_final} final")
print(f"detours    06: 9 out, 6 back  |  08: 7 out, 7 back  |  both: {sorted(d06 & d08)}")
print("ledger balances")

libraries  316 = 268 S1 + 15 S2 + 11 S3 + 18 D1 + 4 D2
isolates   298 entry -> 295 into external QC -> 294 final
detours    06: 9 out, 6 back  |  08: 7 out, 7 back  |  both: ['INF205']
ledger balances


## Palette

Colour carries fate. The three surviving classes take an **ordinal blue ramp** —
darker means more intervention — and the two discard classes are deliberately not
both red: a duplicate morphology is the design working, so it is neutral grey, and
red is reserved for the four genuine losses.

Validated with the data-viz skill's checks (ported to Python — there is no node on
this cluster; the port reproduces the reference palette's published figures exactly,
9.1/19.6 light and 8.4/19.3 dark):

| | light | dark |
|---|---|---|
| ordinal ramp | monotone L, min adjacent ΔL 0.189, hue spread 3°, light end 2.06:1 | same, dark end 2.15:1 |
| worst pair, normal vision | ΔE 17.6 (floor 15) | ΔE 17.0 |
| worst pair, protan/deutan | ΔE 9.1 (target 8) | ΔE 9.1 |

The ramp deliberately spans the lightness band and the grey sits below the chroma
floor, so the *categorical* checks fail it by design — those checks are for
identity palettes, and this is an ordered one plus a reserved neutral.

In [4]:
FATE_RANK = {"S1": 0, "S2": 1, "S3": 2, "D1": 3, "D2": 4}
FATE_FILL = {b: f"f-{b.lower()}" for b in BANDS}   # CSS class — librsvg has no custom properties
LEGEND = {"S1": "straight through", "S2": "sound, reference disagrees",
          "S3": "needed intervention", "D1": "duplicate morphology, no isolate lost",
          "D2": "isolate lost"}

LIGHT = dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", rule="#e1e0d9",
             node="#3f3e3b",
             s1="#86b6ef", s2="#2a78d6", s3="#104281", d1="#898781", d2="#d03b3b")
DARK  = dict(surface="#1a1a19", ink="#ffffff", ink2="#c3c2b7", muted="#898781", rule="#2c2c2a",
             node="#cfcec6",
             s1="#9ec5f4", s2="#3987e5", s3="#184f95", d1="#898781", d2="#d03b3b")

def _rules(p):
    return (f".bg{{fill:{p['surface']};}}"
            f"text{{font-family:system-ui,-apple-system,'Segoe UI',sans-serif;fill:{p['ink2']};}}"
            f".t1{{font-size:19px;font-weight:600;fill:{p['ink']};}}"
            f".t2{{font-size:12.5px;fill:{p['ink2']};}}"
            f".stage{{font-size:11px;font-weight:600;fill:{p['ink']};letter-spacing:0.02em;}}"
            f".stageq{{font-size:10.5px;fill:{p['muted']};}}"
            f".lab{{font-size:11.5px;fill:{p['ink2']};}}"
            f".num{{font-size:13px;font-weight:600;fill:{p['ink']};font-variant-numeric:tabular-nums;}}"
            f".small{{font-size:10.5px;fill:{p['muted']};}}"
            f".rule{{stroke:{p['rule']};stroke-width:1;fill:none;}}"
            f".lead{{stroke:{p['rule']};stroke-width:1;fill:none;}}"
            f".gate{{stroke:{p['rule']};stroke-width:1;fill:none;}}"
            f".nd{{fill:{p['node']};opacity:0.55;}}"
            # Halo: a stroked copy drawn *under* the real glyphs, so a label stays
            # legible where it must sit over a ribbon. Not CSS `paint-order` —
            # librsvg ignores that and then strokes over the fill, erasing the text.
            f".hl{{stroke:{p['surface']};stroke-width:3.5px;stroke-linejoin:round;"
            f"fill:{p['surface']};}}"
            + "".join(f".f-{k}{{fill:{p[k]};}}" for k in ("s1", "s2", "s3", "d1", "d2")))

# Light values are literal so any rasterizer renders them; the dark block is a set
# of steps selected for the dark surface, not an automatic flip of the light one.
CSS = _rules(LIGHT) + "@media (prefers-color-scheme:dark){" + _rules(DARK) + "}"

def esc(s):
    return html.escape(str(s))

def svg(w, h, body):
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="{w}" '
            f'height="{h}" role="img"><style>{CSS}</style>'
            f'<rect width="{w}" height="{h}" class="bg"/>{body}</svg>')

def ribbon(x1, y1, x2, y2, thick, cls, op=1.0):
    'd3-sankey style link: a constant-thickness band on a cubic bezier spine.'
    xm = (x1 + x2) / 2
    return (f'<path class="{cls}" opacity="{op}" d="M{x1:.1f} {y1:.1f} '
            f'C{xm:.1f} {y1:.1f} {xm:.1f} {y2:.1f} {x2:.1f} {y2:.1f} '
            f'L{x2:.1f} {y2 + thick:.1f} '
            f'C{xm:.1f} {y2 + thick:.1f} {xm:.1f} {y1 + thick:.1f} {x1:.1f} {y1 + thick:.1f} Z"/>')

## Layout helpers

`check_no_overlap` is the reason the figures can be trusted without opening them:
text extents are estimated from the glyph metrics of the sans stack and any two
labels sharing a baseline are asserted not to intersect. Both figures run it before
they are written, so a label collision fails the notebook instead of shipping.

In [5]:
CHAR_W = {"t1": 0.60, "t2": 0.52, "stage": 0.58, "stageq": 0.52,
          "lab": 0.52, "num": 0.58, "small": 0.52}
SIZE   = {"t1": 19, "t2": 12.5, "stage": 11.5, "stageq": 10.5,
          "lab": 11.5, "num": 13, "small": 10.5}

def text_extent(s, cls):
    return len(str(s)) * SIZE[cls] * CHAR_W[cls]

def check_no_overlap(labels, tag, w=None):
    'labels: [(text, cls, x, y, anchor)]. Assert no two on a shared baseline collide.'
    boxes = []
    for s, cls, x, y, anchor in labels:
        ext = text_extent(s, cls)
        x0 = x - ext / 2 if anchor == "middle" else (x - ext if anchor == "end" else x)
        boxes.append((y, x0, x0 + ext, s))
    bad = [f"{si!r} <-> {sj!r}"
           for i, (yi, ai, bi, si) in enumerate(boxes)
           for yj, aj, bj, sj in boxes[i + 1:]
           if abs(yi - yj) < 12 and ai < bj - 1 and aj < bi - 1]
    assert not bad, f"{tag}: overlapping labels: {bad}"
    if w is not None:
        out = [(s, round(a, 1), round(b, 1)) for _, a, b, s in boxes if a < 0 or b > w]
        assert not out, f"{tag}: labels overflow the {w}px canvas: {out}"
    return len(boxes)

## Figure 1 — attrition, as a Sankey

Nodes are the states a library can be in; ribbons are libraries moving between them,
drawn at true thickness. What makes it readable is that **each link is decomposed by
the ultimate fate of the libraries it carries** — so the surviving channel keeps its
268 / 15 / 11 structure the whole way across instead of reading as one flat slab, and
a colour can be followed from the entry node to wherever it leaves.

Both detours are structure rather than annotation: `06_asm_fixing` takes 9 out of the
flow and returns 6, and the 7 hybrids `clinopore-nf` could not assemble leave at
`08_hybrid_asm` and all 7 rejoin. The flagged 15 branch out at `07_external_qc` and
come back — that crossing is the point, not a defect.

Ordering is **fate-first at both ends of every link**, so a fate band holds the same
slot in every node it touches and position only breaks ties inside a band. Ordering by
target instead (plain d3-sankey) removes one crossing at the far left but interleaves
the grey discards into the middle of the blue channel, which reads worse.

In [6]:
def build_graph():
    'Nodes and fate-decomposed links. Every count is derived.'
    n_lib    = len(roster)
    pass_int = int(internal.internal_qc_decision.str.startswith("PASS").sum())
    redo     = int(internal.internal_qc_decision.str.startswith("REDO").sum())
    fail_var = int((internal.internal_qc_decision == "FAIL_VARIANT").sum())
    into_ext = len(external)
    recovered, lost_fix = into_ext - pass_int, redo - (into_ext - pass_int)

    ed = external.external_qc_decision.value_counts()
    e_pass, e_flag = int(ed["PASS"]), int(ed["PASS_REFERENCE_MISMATCH"])
    e_dupe, e_lost = int(ed["DROP_VARIANT"]), int(ed["DROP_INTERNAL_MISMATCH"])
    n_repol = int(final.assembly_route.str.contains("repolish").sum())
    n_clin  = len(final) - n_repol

    nodes = [   # id, column, value, label, style ("nd" = a through node)
        ("E",  0, n_lib,      "sequenced libraries",      "nd"),
        ("A1", 1, pass_int,   "self-consistent",          "nd"),
        ("A2", 1, redo,       "recoverable",              "nd"),
        ("A3", 1, fail_var,   "duplicate morphology",     "d1"),
        ("B0", 2, into_ext,   "into external QC",         "nd"),
        ("B3", 2, lost_fix,   "isolate lost",             "d2"),
        ("P0", 3, e_pass,     "reproduces the reference", "nd"),
        ("P1", 3, e_flag,     "reference disagrees",      "nd"),
        ("P2", 3, e_dupe,     "duplicate morphology",     "d1"),
        ("P3", 3, e_lost,     "isolate lost",             "d2"),
        ("H0", 4, n_clin,     "clinopore-nf",             "nd"),
        ("H1", 4, n_repol,    "repolished",               "nd"),
        ("F",  5, len(final), "final genomes",            "nd"),
    ]
    links = trace_paths()
    for nid, _, val, _, _ in nodes:          # every node balances
        for side, got in (("in",  sum(v for _, t, _, v in links if t == nid)),
                          ("out", sum(v for s, _, _, v in links if s == nid))):
            assert got in (0, val), f"{nid} {side}flow {got} != {val}"
    return nodes, links


def trace_paths():
    'Walk every library through the graph, tallying each hop by its fate class.'
    f = final.set_index("library")
    interv  = set(f.index[(f.read_prep != "standard") | f.assembly_route.str.contains("repolish")])
    flagged = set(f.index[~f.matches_reference])
    repol   = set(f.index[f.assembly_route.str.contains("repolish")])
    kept    = set(f.index)
    idec = internal.set_index("sample").internal_qc_decision
    edec = external.set_index("sample").external_qc_decision
    ext_node = {"PASS": "P0", "PASS_REFERENCE_MISMATCH": "P1",
                "DROP_VARIANT": "P2", "DROP_INTERNAL_MISMATCH": "P3"}

    tally = {}
    for s in roster["sample"]:
        fate = ("S3" if s in interv else "S2" if s in flagged else "S1") if s in kept else None
        d = idec[s]
        if d == "FAIL_VARIANT":
            path, fate = ["E", "A3"], "D1"
        elif d.startswith("REDO") and s not in edec.index:
            path, fate = ["E", "A2", "B3"], "D2"          # dropped inside 06_asm_fixing
        else:
            e = ext_node[edec[s]]
            path = ["E", "A2" if d.startswith("REDO") else "A1", "B0", e]
            if e in ("P0", "P1"):
                path += ["H1" if s in repol else "H0", "F"]
            else:
                fate = "D1" if e == "P2" else "D2"
        for a, b in zip(path, path[1:]):
            tally[(a, b, fate)] = tally.get((a, b, fate), 0) + 1
    return [(a, b, fa, v) for (a, b, fa), v in tally.items()]


COLS = [("", ""), ("05_internal_qc", "self-consistent?"), ("06_asm_fixing", "recoverable?"),
        ("07_external_qc", "the labelled isolate?"), ("08_hybrid_asm", "polished how?"), ("", "")]


def figure_1(nodes, links):
    W, U, NODE_W, PAD = 1200, 1.16, 9, 38    # U: px per library — thickness is the true count
    X0, X1, Y_TOP = 176, 1012, 214

    fate_n = {}
    for s, _, fa, v in links:
        if s == "E":
            fate_n[fa] = fate_n.get(fa, 0) + v
    assert sum(fate_n.values()) == 316 and fate_n["S1"] == 268, fate_n

    ncols = max(c for _, c, _, _, _ in nodes) + 1
    colx = {c: X0 + (X1 - X0) * c / (ncols - 1) for c in range(ncols)}
    pos, bottom = {}, {}
    for c in range(ncols):
        y = Y_TOP
        for nid, cc, val, lab, style in nodes:
            if cc == c:
                pos[nid] = dict(x=colx[c], y=y, h=val * U, val=val, lab=lab, style=style, col=c)
                y += val * U + PAD
        bottom[c] = y - PAD
    H = int(max(bottom.values()) + 74)

    order   = {nid: i for i, (nid, *_) in enumerate(nodes)}
    out_cur = {nid: pos[nid]["y"] for nid in pos}
    in_cur  = {nid: pos[nid]["y"] for nid in pos}
    placed  = []
    for s, t, fa, v in sorted(links, key=lambda L: (order[L[0]], FATE_RANK[L[2]], pos[L[1]]["y"])):
        placed.append((s, t, fa, v, out_cur[s]))
        out_cur[s] += v * U
    placed.sort(key=lambda L: (order[L[1]], FATE_RANK[L[2]], pos[L[0]]["y"]))
    drawn = []
    for s, t, fa, v, y1 in placed:
        drawn.append((s, t, fa, v, y1, in_cur[t]))
        in_cur[t] += v * U
    for nid, p in pos.items():
        for cur in (out_cur, in_cur):
            assert cur[nid] <= p["y"] + p["h"] + 1e-6, f"{nid} over-subscribed"

    o, labels = [], []
    def T(s, cls, x, y, anchor="start", halo=False):
        a = f' text-anchor="{anchor}"' if anchor != "start" else ""
        if halo:
            o.append(f'<text class="{cls} hl" x="{x:.1f}" y="{y:.1f}"{a}>{esc(s)}</text>')
        o.append(f'<text class="{cls}" x="{x:.1f}" y="{y:.1f}"{a}>{esc(s)}</text>')
        labels.append((str(s), cls, x, y, anchor))

    T("From sequenced library to final assembly", "t1", 40, 46)
    T("316 sequenced libraries become 294 closed hybrid genomes. Ribbon thickness is the number "
      "of libraries, and every ribbon is coloured by where it ends up.", "t2", 40, 70)

    lx = 40                                   # legend: decodes the ordinal ramp
    for fa in BANDS:
        o.append(f'<rect x="{lx}" y="{Y_TOP - 128}" width="10" height="10" '
                 f'class="{FATE_FILL[fa]}" rx="2"/>')
        T(fate_n[fa], "num", lx + 16, Y_TOP - 119)
        wn = text_extent(fate_n[fa], "num") + 6
        T(LEGEND[fa], "lab", lx + 16 + wn, Y_TOP - 119)
        lx += 16 + wn + text_extent(LEGEND[fa], "lab") + 30

    for c, (name, q) in enumerate(COLS):
        if name:
            x = colx[c] + NODE_W / 2
            T(name, "stage", x, Y_TOP - 54, "middle")
            T(q, "stageq", x, Y_TOP - 39, "middle")
            o.append(f'<line class="rule" x1="{x:.1f}" y1="{Y_TOP - 30}" '
                     f'x2="{x:.1f}" y2="{Y_TOP - 14}"/>')

    for s, t, fa, v, y1, y2 in sorted(drawn, key=lambda L: -L[3]):   # thick first
        o.append(ribbon(pos[s]["x"] + NODE_W, y1, pos[t]["x"], y2, v * U, FATE_FILL[fa], 0.92))
    for nid, p in pos.items():
        cls = "nd" if p["style"] == "nd" else f'f-{p["style"]}'
        o.append(f'<rect x="{p["x"]:.1f}" y="{p["y"]:.1f}" width="{NODE_W}" '
                 f'height="{max(p["h"], 2.5):.1f}" class="{cls}" rx="2"/>')

    # Through nodes label above the bar, where the column is clear; a terminal node
    # is only a few px tall, so its label sits beside it instead.
    for nid, p in pos.items():
        lx = p["x"] + NODE_W + 9
        if p["style"] == "nd":
            T(p["val"], "num", lx, p["y"] - 17, halo=True)
            T(p["lab"], "lab", lx, p["y"] - 4, halo=True)
        else:
            cy = p["y"] + max(p["h"], 2.5) / 2
            T(p["val"], "num", lx, cy - 4, halo=True)
            T(p["lab"], "lab", lx, cy + 11, halo=True)

    T("Both detours rejoin the flow rather than ending it: 06_asm_fixing takes 9 out and returns "
      "6, 08_hybrid_asm takes 7 out and returns all 7.", "small", 40, H - 40)
    T("A duplicate morphology costs no isolate — .1 and .2 are two colony picks of one isolate. "
      "Only the 4 in red cost the collection an isolate.", "small", 40, H - 24)

    check_no_overlap(labels, "figure 1", W)
    return svg(W, H, "".join(o))


nodes, links = build_graph()
svg1 = figure_1(nodes, links)
(HERE / "flow_attrition.svg").write_text(svg1)
print(f"{len(nodes)} nodes, {len(links)} fate-decomposed links")
print(f"wrote flow_attrition.svg  ({len(svg1):,} bytes)")

13 nodes, 27 fate-decomposed links
wrote flow_attrition.svg  (11,837 bytes)


## Figure 2 — provenance of the 294

The counts here span 283 to 1. Nothing survives that range as an area encoding —
a ribbon or a slice for the single Autocycler genome would be a hairline at any
honest scale, and shrinking the other 283 to compensate would be a lie. So this
one carries no area at all: it is the route tree, and the number is the value.

The swatch reuses figure 1's ordinal ramp, so depth means the same thing in both.

In [7]:
# ROUTE_ORDER groups the read preps contiguously — interleaved rows cannot be
# bracketed, and the two group labels collide. The assert below enforces it.
ROUTE_ORDER = ["clinopore", "flye (03_assembly) > repolish", "autocycler > repolish",
               "kraken2 decontam > clinopore"]
ROUTE_DEPTH = {"clinopore": 0, "flye (03_assembly) > repolish": 1,
               "autocycler > repolish": 2, "kraken2 decontam > clinopore": 1}
DEPTH_FILL = {0: "f-s1", 1: "f-s2", 2: "f-s3"}
ROUTE_GLOSS = {
    "clinopore": "Flye → Medaka → Polypolish → POLCA, all of it clinopore-nf",
    "flye (03_assembly) > repolish": "clinopore's Flye stage failed; polished the existing ONT assembly",
    "autocycler > repolish": "Flye cannot resolve INF205; polished the multi-assembler consensus",
    "kraken2 decontam > clinopore": "same as clinopore, but a second organism was removed from the ONT reads first",
}

def figure_2(final):
    W = 1000
    X_READS, X_BRACKET, X_LEAF, X_N = 214, 232, 268, 962
    Y0, ROW = 176, 46

    counts = final.assembly_route.value_counts()
    reads = final.groupby("read_prep").size()
    route_reads = final.groupby("assembly_route").read_prep.unique()
    assert set(counts.index) == set(ROUTE_ORDER), set(counts.index) ^ set(ROUTE_ORDER)
    for r, v in route_reads.items():
        assert len(v) == 1, f"route {r} mixes read preps: {v}"
    rows = [(r, int(counts[r]), route_reads[r][0]) for r in ROUTE_ORDER]
    H = int(Y0 + ROW * len(rows) + 78)

    o, labels = [], []
    def T(s, cls, x, y, anchor="start"):
        a = f' text-anchor="{anchor}"' if anchor != "start" else ""
        o.append(f'<text class="{cls}" x="{x}" y="{y}"{a}>{esc(s)}</text>')
        labels.append((str(s), cls, x, y, anchor))

    T("How each of the 294 genomes was produced", "t1", 40, 44)
    T("Reads, then assembler. The counts span 283 to 1, so nothing here is drawn to scale — "
      "the number is the value.", "t2", 40, 68)
    T("ONT READS", "small", X_READS, Y0 - 22, "end")
    T("ASSEMBLY ROUTE", "small", X_LEAF, Y0 - 22)
    T("GENOMES", "small", X_N, Y0 - 22, "end")
    o.append(f'<line class="gate" x1="40" y1="{Y0 - 14}" x2="{W - 38}" y2="{Y0 - 14}"/>')

    grouped = {}
    for i, (_, _, prep) in enumerate(rows):
        grouped.setdefault(prep, []).append(i)
    for prep, idxs in grouped.items():
        assert idxs == list(range(idxs[0], idxs[-1] + 1)), \
            f"read-prep group {prep!r} is not contiguous in ROUTE_ORDER: rows {idxs}"
        yc = (Y0 + ROW * idxs[0] + Y0 + ROW * idxs[-1]) / 2
        T(int(reads[prep]), "num", X_READS, yc - 4, "end")
        T(prep, "lab", X_READS, yc + 11, "end")
        if len(idxs) > 1:
            o.append(f'<path class="lead" d="M{X_BRACKET} {Y0 + ROW * idxs[0]} '
                     f'V{Y0 + ROW * idxs[-1]}"/>')
        for i in idxs:
            o.append(f'<path class="lead" d="M{X_BRACKET} {Y0 + ROW * i} H{X_LEAF - 22}"/>')

    for i, (route, cnt, _) in enumerate(rows):
        y = Y0 + ROW * i
        o.append(f'<rect x="{X_LEAF - 14}" y="{y - 5}" width="10" height="10" '
                 f'class="{DEPTH_FILL[ROUTE_DEPTH[route]]}" rx="2"/>')
        T(route, "lab", X_LEAF, y - 2)
        T(ROUTE_GLOSS[route], "small", X_LEAF, y + 13)
        T(cnt, "num", X_N, y + 2, "end")

    yb = Y0 + ROW * len(rows)
    o.append(f'<line class="gate" x1="40" y1="{yb - 16}" x2="{W - 38}" y2="{yb - 16}"/>')
    T("total", "lab", X_LEAF, yb + 6)
    T(int(counts.sum()), "num", X_N, yb + 6, "end")
    T("Swatch depth is how much intervention the genome needed — the same ramp the attrition "
      "figure uses. 283 of 294 needed nothing beyond the standard pipeline.", "small", 40, yb + 44)

    assert int(counts.sum()) == len(final) == 294
    check_no_overlap(labels, "figure 2", W)
    return svg(W, H, "".join(o))

svg2 = figure_2(final)
(HERE / "route_provenance.svg").write_text(svg2)
print(f"wrote route_provenance.svg  ({len(svg2):,} bytes)")

wrote route_provenance.svg  (4,298 bytes)


## What was checked

Both figures were written only after the ledger balanced and the label geometry
was measured. If a stage table changes and the counts stop reconciling — or a
renamed route makes a label wider than the space it has — this notebook raises
rather than writing a figure that disagrees with its own data.

In [8]:
for f_ in ("flow_attrition.svg", "route_provenance.svg"):
    p = HERE / f_
    assert p.exists() and p.stat().st_size > 2000
    print(f"{f_:24} {p.stat().st_size:>7,} bytes")
print()
print("mass conservation   316 = 268 + 15 + 11 + 18 + 4, and 294 kept")
print("survivor partition  no genome is both flagged and intervened")
print("detours             06: 9/6 back, 08: 7/7 back, INF205 in both")
print("label geometry      no shared-baseline overlaps, nothing off-canvas")

flow_attrition.svg        11,839 bytes
route_provenance.svg       4,308 bytes

mass conservation   316 = 268 + 15 + 11 + 18 + 4, and 294 kept
survivor partition  no genome is both flagged and intervened
detours             06: 9/6 back, 08: 7/7 back, INF205 in both
label geometry      no shared-baseline overlaps, nothing off-canvas
